# ALIP – Tier 2: LayoutLMv3 fine-tuning (Colab)

**Runtime → Change runtime type → T4 GPU**, then run all cells (~15 min).

Trains on the 109 synthetic `train` docs, early-stops on the 27 `val` docs, predicts the 34-doc held-out `test` split (24 synthetic + 10 real), and downloads `layoutlmv3_fine_tuned__test.json`. Put that file in `results/predictions/` and run `python compare_models.py`.

In [ ]:
!git clone -q https://github.com/SimbaPonde/ALIP-Mineral-License-Classification.git
%cd ALIP-Mineral-License-Classification
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install pytesseract pdf2image pdfplumber

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4')

## 1. Dataset split + OCR (~3 min)

In [ ]:
!python build_unified_dataset.py
!python layoutlmv3_pipeline.py ocr | tail -3

## 2. Fine-tune (early stopping on val macro-F1)

In [ ]:
!python layoutlmv3_pipeline.py train --epochs 15 --lr 1e-5 --batch-size 4 --patience 4

Optional: save the checkpoint to Drive so a session reset doesn't lose it.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r models/layoutlmv3_best /content/drive/MyDrive/ALIP_layoutlmv3_best

## 3. Predict held-out test split

In [ ]:
!python layoutlmv3_pipeline.py predict --split test

In [ ]:
import json
d = json.load(open('results/predictions/layoutlmv3_fine_tuned__test.json'))
err = [r for r in d['predictions'] if r['true'] != r['pred']]
real = [r for r in d['predictions'] if r['source'] == 'real']
print(f"test acc {1 - len(err)/len(d['predictions']):.3f} | real {sum(r['true']==r['pred'] for r in real)}/{len(real)}")
for r in err: print(r['source'], r['doc_id'], r['true'], '->', r['pred'])

In [ ]:
!python compare_models.py

In [ ]:
from google.colab import files
files.download('results/predictions/layoutlmv3_fine_tuned__test.json')